# Key Insights — Final Fact Table Validation

## Overview
The final sales fact dataset was successfully validated after feature engineering and dimensional enrichment.

The dataset now contains:
- transactional sales metrics
- profitability KPIs
- product hierarchy dimensions
- customer dimensions
- territory dimensions
- time-based analytical features

This transformed dataset represents a business-ready analytical model suitable for downstream analytics workflows.

---

# Major Insights

## 1. Business KPIs Successfully Engineered
The dataset now includes:
- revenue metrics
- profitability metrics
- margin calculations
- pricing analytics

These metrics support:
- executive KPI reporting
- product analysis
- profitability analysis

---

## 2. Dimensional Enrichment Completed
The fact dataset now integrates:
- product hierarchy
- customer identifiers
- territory information
- time-series dimensions

This significantly improves analytical depth and reporting capability.

---

## 3. Time-Series Readiness Achieved
Date-based features now support:
- monthly analysis
- quarterly analysis
- yearly trend analysis
- seasonality reporting

---

## 4. Dataset Ready for SQL Analytics
The final dataset is now structured appropriately for:
- SQL-based business questions
- dimensional modeling
- Tableau dashboards
- advanced analytics

---

# Overall Conclusion

The project has successfully transformed raw operational data into a structured analytical fact-style dataset.

This establishes the foundation for the next project phases:
- fact & dimension table creation
- SQL analytical modeling
- dashboard development
- advanced business KPI analysis

In [ ]:

# Import Libraries

import pandas as pd
import numpy as np

# Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
base_path = '/Users/yashmalviya/AdventureWorks-Docker/01_data/02_processed/'

sales_order_header = pd.read_csv(
    base_path + 'sales_order_header_cleaned.csv'
)

sales_order_detail = pd.read_csv(
    base_path + 'sales_order_detail_cleaned.csv'
)

customer = pd.read_csv(
    base_path + 'customer_cleaned.csv'
)

product = pd.read_csv(
    base_path + 'product_cleaned.csv'
)

product_category = pd.read_csv(
    base_path + 'product_category_cleaned.csv'
)

product_subcategory = pd.read_csv(
    base_path + 'product_subcategory_cleaned.csv'
)

sales_territory = pd.read_csv(
    base_path + 'sales_territory_cleaned.csv'
)

In [ ]:

# Create Product Hierarchy

product_enriched = product.merge(
    product_subcategory[
        [
            'productsubcategoryid',
            'name',
            'productcategoryid'
        ]
    ],
    on='productsubcategoryid',
    how='left',
    suffixes=('', '_subcategory')
)

# Rename Subcategory Column
product_enriched.rename(
    columns={
        'name_subcategory': 'product_subcategory_name'
    },
    inplace=True
)

# Merge Product Category

product_enriched = product_enriched.merge(
    product_category[
        [
            'productcategoryid',
            'name'
        ]
    ],
    on='productcategoryid',
    how='left',
    suffixes=('', '_category')
)

# Rename Category Column
product_enriched.rename(
    columns={
        'name_category': 'product_category_name'
    },
    inplace=True
)

# Preview Product Hierarchy

product_enriched[
    [
        'productid',
        'name',
        'product_subcategory_name',
        'product_category_name'
    ]
].head()

,productid,name,product_subcategory_name,product_category_name
0,1,Adjustable Race,NaN,NaN
1,2,Bearing Ball,NaN,NaN
2,3,BB Ball Bearing,NaN,NaN
3,4,Headset Ball Bearings,NaN,NaN
4,316,Blade,NaN,NaN


In [17]:
product_enriched[
    product_enriched['product_category_name'].notnull()
][
    [
        'productid',
        'name',
        'product_subcategory_name',
        'product_category_name'
    ]
].head(10)

,productid,name,product_subcategory_name,product_category_name
209,680,"HL Road Frame - Black, 58",Road Frames,Components
210,706,"HL Road Frame - Red, 58",Road Frames,Components
211,707,"Sport-100 Helmet, Red",Helmets,Accessories
212,708,"Sport-100 Helmet, Black",Helmets,Accessories
213,709,"Mountain Bike Socks, M",Socks,Clothing
214,710,"Mountain Bike Socks, L",Socks,Clothing
215,711,"Sport-100 Helmet, Blue",Helmets,Accessories
216,712,AWC Logo Cap,Caps,Clothing
217,713,"Long-Sleeve Logo Jersey, S",Jerseys,Clothing
218,714,"Long-Sleeve Logo Jersey, M",Jerseys,Clothing


In [ ]:

# Convert Date Columns

sales_order_header['orderdate'] = pd.to_datetime(
    sales_order_header['orderdate']
)

sales_order_header['duedate'] = pd.to_datetime(
    sales_order_header['duedate']
)

sales_order_header['shipdate'] = pd.to_datetime(
    sales_order_header['shipdate']
)

# Create Date Features

sales_order_header['order_year'] = (
    sales_order_header['orderdate'].dt.year
)

sales_order_header['order_month'] = (
    sales_order_header['orderdate'].dt.month
)

sales_order_header['order_month_name'] = (
    sales_order_header['orderdate'].dt.month_name()
)

sales_order_header['order_quarter'] = (
    sales_order_header['orderdate'].dt.quarter
)

sales_order_header['order_day_name'] = (
    sales_order_header['orderdate'].dt.day_name()
)

# Create Online Order Flag

sales_order_header['is_online_order'] = (
    sales_order_header['salespersonid']
    .isnull()
    .astype(int)
)

In [ ]:
# =========================================================
# Create Sales Fact Table
# =========================================================

fact_sales = sales_order_detail.merge(
    sales_order_header[
        [
            'salesorderid',
            'customerid',
            'territoryid',
            'orderdate',
            'order_year',
            'order_month',
            'order_month_name',
            'order_quarter',
            'order_day_name',
            'is_online_order'
        ]
    ],
    on='salesorderid',
    how='left'
)

# Merge Product Hierarchy

fact_sales = fact_sales.merge(
    product_enriched[
        [
            'productid',
            'name',
            'product_subcategory_name',
            'product_category_name',
            'standardcost',
            'listprice'
        ]
    ],
    on='productid',
    how='left'
)

# Rename Product Name
fact_sales.rename(
    columns={
        'name': 'product_name'
    },
    inplace=True
)

# Create Business Metrics

# Revenue
fact_sales['revenue'] = (
    fact_sales['linetotal']
)

# Total Cost
fact_sales['total_cost'] = (
    fact_sales['standardcost'] *
    fact_sales['orderqty']
)

# Profit
fact_sales['profit'] = (
    fact_sales['revenue'] -
    fact_sales['total_cost']
)

# Profit Margin %
fact_sales['profit_margin_pct'] = (
    (
        fact_sales['profit'] /
        fact_sales['revenue']
    ) * 100
).round(2)

# Preview Final Fact Table

fact_sales.head()

,salesorderid,salesorderdetailid,carriertrackingnumber,orderqty,productid,specialofferid,unitprice,unitpricediscount,linetotal,rowguid,modifieddate,customerid,territoryid,orderdate,order_year,order_month,order_month_name,order_quarter,order_day_name,is_online_order,product_name,product_subcategory_name,product_category_name,standardcost,listprice,revenue,total_cost,profit,profit_margin_pct
0,43659,1,4911-403C-98,1,776,1,2024.994,0.0,2024.994,b207c96d-d9e6-402b-8470-2cc176c42283,2011-05-31 00:00:00.000,29825,5,2011-05-31,2011,5,May,2,Tuesday,0,"Mountain-100 Black, 42",Mountain Bikes,Bikes,1898.0944,3374.99,2024.994,1898.0944,126.8996,6.27
1,43659,2,4911-403C-98,3,777,1,2024.994,0.0,6074.982,7abb600d-1e77-41be-9fe5-b9142cfc08fa,2011-05-31 00:00:00.000,29825,5,2011-05-31,2011,5,May,2,Tuesday,0,"Mountain-100 Black, 44",Mountain Bikes,Bikes,1898.0944,3374.99,6074.982,5694.2832,380.6988,6.27
2,43659,3,4911-403C-98,1,778,1,2024.994,0.0,2024.994,475cf8c6-49f6-486e-b0ad-afc6a50cdd2f,2011-05-31 00:00:00.000,29825,5,2011-05-31,2011,5,May,2,Tuesday,0,"Mountain-100 Black, 48",Mountain Bikes,Bikes,1898.0944,3374.99,2024.994,1898.0944,126.8996,6.27
3,43659,4,4911-403C-98,1,771,1,2039.994,0.0,2039.994,04c4de91-5815-45d6-8670-f462719fbce3,2011-05-31 00:00:00.000,29825,5,2011-05-31,2011,5,May,2,Tuesday,0,"Mountain-100 Silver, 38",Mountain Bikes,Bikes,1912.1544,3399.99,2039.994,1912.1544,127.8396,6.27
4,43659,5,4911-403C-98,1,772,1,2039.994,0.0,2039.994,5a74c7d2-e641-438e-a7ac-37bf23280301,2011-05-31 00:00:00.000,29825,5,2011-05-31,2011,5,May,2,Tuesday,0,"Mountain-100 Silver, 42",Mountain Bikes,Bikes,1912.1544,3399.99,2039.994,1912.1544,127.8396,6.27


In [ ]:

# Create Dimension Tables

# Product Dimension
dim_product = product_enriched[
    [
        'productid',
        'name',
        'product_subcategory_name',
        'product_category_name',
        'standardcost',
        'listprice',
        'color',
        'size',
        'class',
        'style'
    ]
].copy()

# Rename Product Name
dim_product.rename(
    columns={
        'name': 'product_name'
    },
    inplace=True
)


# Customer Dimension

dim_customer = customer.copy()

# Territory Dimension

dim_territory = sales_territory.copy()

# Date Dimension

dim_date = sales_order_header[
    [
        'orderdate',
        'order_year',
        'order_month',
        'order_month_name',
        'order_quarter',
        'order_day_name'
    ]
].drop_duplicates()

# Preview Dimensions

print("dim_product:", dim_product.shape)
print("dim_customer:", dim_customer.shape)
print("dim_territory:", dim_territory.shape)
print("dim_date:", dim_date.shape)

dim_product: (504, 10)
dim_customer: (19820, 8)
dim_territory: (10, 10)
dim_date: (1124, 6)


# Final Analytical Dataset Export

## Objective
The purpose of this section is to export the final analytical fact and dimension tables for downstream business intelligence and SQL analytics workflows.

The exported datasets represent the final analytical layer of the project.

These datasets will be used for:
- SQL business analysis
- Power BI dashboards
- KPI reporting
- executive analytics
- dimensional modeling

---

# Final Analytical Tables

The following analytical tables will be exported:

| Table | Type |
|---|---|
| fact_sales | Fact Table |
| dim_product | Dimension Table |
| dim_customer | Dimension Table |
| dim_territory | Dimension Table |
| dim_date | Dimension Table |

---

# Business Importance

Separating data into:
- fact tables
- dimension tables

improves:
- query performance
- dashboard scalability
- business readability
- analytical flexibility

This structure closely resembles real-world star-schema warehouse modeling.

---

# Expected Outcome

At the end of this section:
- the final analytical datasets will be exported
- the project will be ready for SQL analytics
- the project will become dashboard-ready

In [ ]:

# Export Final Analytical Tables

base_path = '/Users/yashmalviya/AdventureWorks-Docker/01_data/03_exports/'

# Export Fact Table

fact_sales.to_csv(
    base_path + 'fact_sales.csv',
    index=False
)

# Export Dimension Tables

dim_product.to_csv(
    base_path + 'dim_product.csv',
    index=False
)

dim_customer.to_csv(
    base_path + 'dim_customer.csv',
    index=False
)

dim_territory.to_csv(
    base_path + 'dim_territory.csv',
    index=False
)

dim_date.to_csv(
    base_path + 'dim_date.csv',
    index=False
)

# Export Validation

print("=" * 70)
print("FINAL ANALYTICAL TABLES EXPORTED SUCCESSFULLY")
print("=" * 70)

print("\nExported Tables:")

print("\n1. fact_sales.csv")
print("2. dim_product.csv")
print("3. dim_customer.csv")
print("4. dim_territory.csv")
print("5. dim_date.csv")

print("\nExport Location:")
print(base_path)

FINAL ANALYTICAL TABLES EXPORTED SUCCESSFULLY

Exported Tables:

1. fact_sales.csv
2. dim_product.csv
3. dim_customer.csv
4. dim_territory.csv
5. dim_date.csv

Export Location:
/Users/yashmalviya/AdventureWorks-Docker/01_data/03_exports/
